# 07_event_based_device_control: Notebook Scaffold and Device-Level Data Pipeline

This notebook establishes the complete data pipeline from raw CSVs through 15-minute aligned device-level and room-level features for event-based control.

In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
%matplotlib inline

# Resolve backend/notebooks directory for imports
sys.path.append(os.path.abspath("."))
from utils import (  # noqa: E402
    EVENT_CONTROL_REPORT_METRIC_COLUMNS,
    EventControlConfig,
    augment_event_control_report,
    build_device_level_features,
    build_event_control_comparison_timeseries,
    build_event_features,
    build_room_level_features,
    load_model_artifacts,
    load_processed,
    load_space_events,
    optimize_event_control_schedules,
    save_event_control_schedule_report,
    save_model_artifacts,
)

# Flag to load existing model artifacts
LOAD_EXISTING_MODEL = True
if "LOAD_EXISTING_MODEL" in os.environ:
    LOAD_EXISTING_MODEL = os.environ["LOAD_EXISTING_MODEL"].lower() in ("true", "1", "yes")

## 1. Configurations

In [2]:
comfort_config = {
    "t_min_occupied_celsius": 20.0,
    "t_max_occupied_celsius": 24.0,
    "t_min_unoccupied_celsius": 16.0,
    "t_max_unoccupied_celsius": 28.0,
    "relative_humidity_min_percent": 30.0,
    "relative_humidity_max_percent": 60.0,
    # S02 optimizer comfort bands used when judging event-start feasibility.
    "heating_event_min_celsius": 20.0,
    "heating_event_max_celsius": 24.0,
    "cooling_event_min_celsius": 20.0,
    "cooling_event_max_celsius": 24.0,
}

control_config = {
    "preconditioning_window_hours": 2.0,
    "standby_baseline_heating": 1.2,
    "standby_baseline_cooling": 2.0,
    # S02 candidate space: 15-minute start offsets across the preconditioning window,
    # device subsets capped at four devices, and compact top-candidate diagnostics.
    "candidate_offsets_minutes": list(range(0, 2 * 60 + 1, 15)),
    "max_simultaneous_devices": 4,
    "expected_devices": 4,
    "merge_threshold_minutes": 60.0,
    "top_candidate_count": 5,
    # Bound notebook smoke execution to the first validation week for each season.
    "validation_horizon_days": 7,
}

energy_config = {
    "electricity_price_eur_per_kWh": 0.35,
    "co2_kg_per_kWh": 0.40,
}

## 2. Data Loading & Feature Engineering

In [3]:
# Load seasonal processed datasets
df_heating_raw = load_processed("heating")
df_cooling_raw = load_processed("cooling")

# Load space events
events_heating = load_space_events("heating")
events_cooling = load_space_events("cooling")

# Build device-level features (wide format)
df_dev_heating = build_device_level_features(
    df_heating_raw, standby_baseline=control_config["standby_baseline_heating"]
)
df_dev_cooling = build_device_level_features(
    df_cooling_raw, standby_baseline=control_config["standby_baseline_cooling"]
)

# Build room-level features
df_room_heating = build_room_level_features(df_heating_raw)
df_room_cooling = build_room_level_features(df_cooling_raw)

# Build event features
df_evt_heating = build_event_features(
    df_room_heating.index,
    events_heating,
    preconditioning_window_hours=control_config["preconditioning_window_hours"],
)
df_evt_cooling = build_event_features(
    df_room_cooling.index,
    events_cooling,
    preconditioning_window_hours=control_config["preconditioning_window_hours"],
)

# Merge all features
df_heating_final = pd.concat([df_room_heating, df_dev_heating, df_evt_heating], axis=1)
df_cooling_final = pd.concat([df_room_cooling, df_dev_cooling, df_evt_cooling], axis=1)

print("Heating Final Data Shape:", df_heating_final.shape)
print("Cooling Final Data Shape:", df_cooling_final.shape)

Heating Final Data Shape: (673, 25)
Cooling Final Data Shape: (673, 25)


## 3. Data Quality Summary

In [4]:
def print_quality_summary(df_raw, df_final, events, name):
    print(f"=== {name.upper()} SEASON ===")
    print(f"Raw row count: {len(df_raw)}")
    print(f"Final aligned row count (15-min): {len(df_final)}")
    print(f"Unique devices: {df_raw['device_id'].nunique()}")
    print(f"Event counts: {len(events)}")
    print("Missing value statistics in critical fields:")
    critical_cols = [
        "room_temp",
        "room_return_temp",
        "room_humidity",
        "room_co2",
        "room_power_kw",
        "outside_temperature",
        "event_active",
    ]
    for col in critical_cols:
        missing = df_final[col].isna().sum()
        print(f"  {col}: {missing} missing ({missing / len(df_final) * 100:.2f}%)")
    print()


print_quality_summary(df_heating_raw, df_heating_final, events_heating, "heating")
print_quality_summary(df_cooling_raw, df_cooling_final, events_cooling, "cooling")

=== HEATING SEASON ===
Raw row count: 2692
Final aligned row count (15-min): 673
Unique devices: 4
Event counts: 13
Missing value statistics in critical fields:
  room_temp: 0 missing (0.00%)
  room_return_temp: 0 missing (0.00%)
  room_humidity: 0 missing (0.00%)
  room_co2: 0 missing (0.00%)
  room_power_kw: 0 missing (0.00%)
  outside_temperature: 0 missing (0.00%)
  event_active: 0 missing (0.00%)

=== COOLING SEASON ===
Raw row count: 2692
Final aligned row count (15-min): 673
Unique devices: 4
Event counts: 15
Missing value statistics in critical fields:
  room_temp: 0 missing (0.00%)
  room_return_temp: 0 missing (0.00%)
  room_humidity: 0 missing (0.00%)
  room_co2: 0 missing (0.00%)
  room_power_kw: 0 missing (0.00%)
  outside_temperature: 0 missing (0.00%)
  event_active: 0 missing (0.00%)



## 4. Grey-Box Thermal Model Training & Validation

We fit a grey-box thermal model of the room temperature dynamics for both heating and cooling seasons.

In [5]:
import os

import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from utils import (
    predict_device_power,
    predict_temperature,
    train_device_power_model,
    train_thermal_model,
)

# Splits
split_heat = pd.Timestamp("2026-04-03 00:00:00")
split_cool = pd.Timestamp("2026-05-28 00:00:00")

print(f"Heating split timestamp: {split_heat}")
print(f"Cooling split timestamp: {split_cool}")

artifacts_dir = "../artifacts/event_device_control/models"
required_files = [
    "thermal_model.joblib",
    "scaler.joblib",
    "feature_columns.json",
    "model_metadata.json",
    "device_power_model.joblib",
    "heating_thermal_model.joblib",
    "heating_scaler.joblib",
    "heating_feature_columns.json",
    "heating_model_metadata.json",
    "heating_device_power_model.joblib",
    "cooling_thermal_model.joblib",
    "cooling_scaler.joblib",
    "cooling_feature_columns.json",
    "cooling_model_metadata.json",
    "cooling_device_power_model.joblib",
]
all_exist = all(os.path.exists(os.path.join(artifacts_dir, f)) for f in required_files)

df_h = df_heating_final.copy()
df_c = df_cooling_final.copy()

if "LOAD_EXISTING_MODEL" in globals() and LOAD_EXISTING_MODEL and all_exist:
    print("Loading existing models from disk...")
    model_h, model_power_h = load_model_artifacts(artifacts_dir, "heating")
    model_c, model_power_c = load_model_artifacts(artifacts_dir, "cooling")
else:
    print("Training models from scratch...")
    # Train heating thermal model
    print("\n--- Training Heating Thermal Model ---")
    model_h = train_thermal_model(df_h, "heating", split_heat)

    # Train cooling thermal model
    print("\n--- Training Cooling Thermal Model ---")
    model_c = train_thermal_model(df_c, "cooling", split_cool)

    # Train heating device power model
    print("\n--- Training Heating Device Power Model ---")
    model_power_h = train_device_power_model(df_h, "heating", split_heat)

    # Train cooling device power model
    print("\n--- Training Cooling Device Power Model ---")
    model_power_c = train_device_power_model(df_c, "cooling", split_cool)

print("\n--- Heating Thermal Model Coefficients ---")
print(f"Is fallback: {model_h['is_fallback']}")
print(f"a = {model_h['a']:.6f}")
print(f"b = {model_h['b']:.6f}")
print(f"c_heat = {model_h['c_heat']:.6f}")
print(f"d = {model_h['d']:.6f}")
print(f"intercept = {model_h['intercept']:.6f}")

print("\n--- Cooling Thermal Model Coefficients ---")
print(f"Is fallback: {model_c['is_fallback']}")
print(f"a = {model_c['a']:.6f}")
print(f"b = {model_c['b']:.6f}")
print(f"c_cool = {model_c['c_cool']:.6f}")
print(f"d = {model_c['d']:.6f}")
print(f"intercept = {model_c['intercept']:.6f}")

print("\n--- Heating Device Power Model Parameters ---")
for dev, params in model_power_h["device_params"].items():
    print(
        f"  {dev}: active_power = {params['active_power']:.4f} kW, "
        f"standby_power = {params['standby_power']:.4f} kW"
    )

print("\n--- Cooling Device Power Model Parameters ---")
for dev, params in model_power_c["device_params"].items():
    print(
        f"  {dev}: active_power = {params['active_power']:.4f} kW, "
        f"standby_power = {params['standby_power']:.4f} kW"
    )

Heating split timestamp: 2026-04-03 00:00:00
Cooling split timestamp: 2026-05-28 00:00:00
Loading existing models from disk...

--- Heating Thermal Model Coefficients ---
Is fallback: False
a = 0.980334
b = -0.003233
c_heat = 0.000257
d = 0.111941
intercept = 0.299099

--- Cooling Thermal Model Coefficients ---
Is fallback: False
a = 0.944139
b = 0.026055
c_cool = 0.010398
d = -0.005536
intercept = 0.716733

--- Heating Device Power Model Parameters ---
  device_1: active_power = 17.7203 kW, standby_power = 1.1712 kW
  device_2: active_power = 4.6875 kW, standby_power = 1.1576 kW
  device_3: active_power = 3.9449 kW, standby_power = 1.1711 kW
  device_4: active_power = 5.6901 kW, standby_power = 1.1972 kW

--- Cooling Device Power Model Parameters ---
  device_1: active_power = 2.6264 kW, standby_power = 1.9925 kW
  device_2: active_power = 6.8487 kW, standby_power = 1.8991 kW
  device_3: active_power = 6.8491 kW, standby_power = 1.9736 kW
  device_4: active_power = 2.4398 kW, standby_

### Validation & Accuracy Metrics

In [6]:
# Validation split metrics
val_h = df_h[df_h.index >= split_heat].copy()
val_h["T_next"] = val_h["room_temp"].shift(-1)
val_h = val_h.dropna(subset=["T_next"])
preds_h = predict_temperature(model_h, val_h)
mae_h = mean_absolute_error(val_h["T_next"], preds_h)
rmse_h = root_mean_squared_error(val_h["T_next"], preds_h)

val_c = df_c[df_c.index >= split_cool].copy()
val_c["T_next"] = val_c["room_temp"].shift(-1)
val_c = val_c.dropna(subset=["T_next"])
preds_c = predict_temperature(model_c, val_c)
mae_c = mean_absolute_error(val_c["T_next"], preds_c)
rmse_c = root_mean_squared_error(val_c["T_next"], preds_c)

print("HEATING VALIDATION METRICS:")
print(f"  MAE: {mae_h:.6f} °C")
print(f"  RMSE: {rmse_h:.6f} °C")

print("\nCOOLING VALIDATION METRICS:")
print(f"  MAE: {mae_c:.6f} °C")
print(f"  RMSE: {rmse_c:.6f} °C")

# Check validation MAE boundary
max_validation_mae_celsius = 2.0
assert mae_h < max_validation_mae_celsius, f"Heating MAE too high: {mae_h:.4f}"
assert mae_c < max_validation_mae_celsius, f"Cooling MAE too high: {mae_c:.4f}"
print("\nValidation bounds check passed (MAE < 2.0 °C)!")

# Evaluate device power model predictions on validation set
print("\nHEATING DEVICE POWER RMSE:")
preds_power_h = predict_device_power(model_power_h, val_h)
for i in range(1, 5):
    actual = val_h[f"device_{i}_power_kw"]
    pred = preds_power_h[f"device_{i}_predicted_power_kw"]
    mask = actual.notna() & pred.notna()
    rmse = np.sqrt(np.mean((actual[mask] - pred[mask]) ** 2)) if mask.any() else 0.0
    print(f"  device_{i} RMSE: {rmse:.6f} kW")

print("\nCOOLING DEVICE POWER RMSE:")
preds_power_c = predict_device_power(model_power_c, val_c)
for i in range(1, 5):
    actual = val_c[f"device_{i}_power_kw"]
    pred = preds_power_c[f"device_{i}_predicted_power_kw"]
    mask = actual.notna() & pred.notna()
    rmse = np.sqrt(np.mean((actual[mask] - pred[mask]) ** 2)) if mask.any() else 0.0
    print(f"  device_{i} RMSE: {rmse:.6f} kW")

HEATING VALIDATION METRICS:
  MAE: 0.120819 °C
  RMSE: 0.153426 °C

COOLING VALIDATION METRICS:
  MAE: 0.092220 °C
  RMSE: 0.165202 °C

Validation bounds check passed (MAE < 2.0 °C)!

HEATING DEVICE POWER RMSE:
  device_1 RMSE: 13.138499 kW
  device_2 RMSE: 2.743310 kW
  device_3 RMSE: 2.554929 kW
  device_4 RMSE: 3.175196 kW

COOLING DEVICE POWER RMSE:
  device_1 RMSE: 1.254459 kW
  device_2 RMSE: 0.672106 kW
  device_3 RMSE: 0.651111 kW
  device_4 RMSE: 1.216688 kW


### Save & Reload Verification

In [7]:
# Save model artifacts using helper function if they were not loaded
artifacts_dir = "../artifacts/event_device_control/models"
os.makedirs(artifacts_dir, exist_ok=True)

if not ("LOAD_EXISTING_MODEL" in globals() and LOAD_EXISTING_MODEL and all_exist):
    print("Saving model artifacts...")
    save_model_artifacts(
        artifacts_dir=artifacts_dir,
        thermal_model_dict=model_h,
        device_power_model_dict=model_power_h,
        metrics={"mae": mae_h, "rmse": rmse_h},
        train_timestamps={"split": str(split_heat)},
    )
    save_model_artifacts(
        artifacts_dir=artifacts_dir,
        thermal_model_dict=model_c,
        device_power_model_dict=model_power_c,
        metrics={"mae": mae_c, "rmse": rmse_c},
        train_timestamps={"split": str(split_cool)},
    )
    print(f"Saved models to {artifacts_dir}")
else:
    print("Skipping save since existing models were loaded.")

# Load verifying using load_model_artifacts
model_h_reloaded, model_power_h_reloaded = load_model_artifacts(artifacts_dir, "heating")
model_c_reloaded, model_power_c_reloaded = load_model_artifacts(artifacts_dir, "cooling")

# Verify predictions match exactly
preds_h_reloaded = predict_temperature(model_h_reloaded, val_h)
preds_c_reloaded = predict_temperature(model_c_reloaded, val_c)

assert preds_h_reloaded.equals(preds_h), "Heating reloaded model predictions mismatch!"
assert preds_c_reloaded.equals(preds_c), "Cooling reloaded model predictions mismatch!"

preds_power_h_reloaded = predict_device_power(model_power_h_reloaded, val_h)
preds_power_c_reloaded = predict_device_power(model_power_c_reloaded, val_c)

assert preds_power_h_reloaded.equals(preds_power_h), (
    "Heating power reloaded model predictions mismatch!"
)
assert preds_power_c_reloaded.equals(preds_power_c), (
    "Cooling power reloaded model predictions mismatch!"
)

print("Model save and reload verification passed successfully!")

Skipping save since existing models were loaded.
Model save and reload verification passed successfully!


## 5. Event-Based Optimizer and Control Schedule Reports

Run the event-level optimizer on one validation week for each season, display the selected control schedules plus compact candidate diagnostics, and persist the selected schedule CSVs for downstream S03 reporting.


In [8]:
# Build validation event slices aligned with the S01/S02 model validation splits.
def validation_events_for_week(events_df, split_timestamp, horizon_days):
    validation_end = split_timestamp + pd.Timedelta(days=horizon_days)
    validation_events = events_df[
        (events_df["starts_at"] >= split_timestamp) & (events_df["starts_at"] < validation_end)
    ].copy()
    return validation_events.sort_values(["starts_at", "ends_at"]).reset_index(drop=True)


validation_horizon_days = control_config["validation_horizon_days"]
events_heating_validation = validation_events_for_week(
    events_heating, split_heat, validation_horizon_days
)
events_cooling_validation = validation_events_for_week(
    events_cooling, split_cool, validation_horizon_days
)

df_h_optimizer = df_h[
    (df_h.index >= split_heat)
    & (df_h.index < split_heat + pd.Timedelta(days=validation_horizon_days))
].copy()
df_c_optimizer = df_c[
    (df_c.index >= split_cool)
    & (df_c.index < split_cool + pd.Timedelta(days=validation_horizon_days))
].copy()

print(
    f"Heating optimizer frame: {df_h_optimizer.index.min()} to "
    f"{df_h_optimizer.index.max()} ({len(df_h_optimizer)} rows)"
)
print(
    f"Cooling optimizer frame: {df_c_optimizer.index.min()} to "
    f"{df_c_optimizer.index.max()} ({len(df_c_optimizer)} rows)"
)
print(f"Heating validation events: {len(events_heating_validation)}")
print(f"Cooling validation events: {len(events_cooling_validation)}")

if events_heating_validation.empty:
    raise ValueError("No heating validation events found in the configured S02 validation week.")
if events_cooling_validation.empty:
    raise ValueError("No cooling validation events found in the configured S02 validation week.")

Heating optimizer frame: 2026-04-03 00:00:00 to 2026-04-06 00:00:00 (289 rows)
Cooling optimizer frame: 2026-05-28 00:00:00 to 2026-06-01 00:00:00 (385 rows)
Heating validation events: 6
Cooling validation events: 10


### Optimize Heating Validation Events


In [9]:
heating_event_config = EventControlConfig(
    mode="heating",
    min_temp=comfort_config["heating_event_min_celsius"],
    max_temp=comfort_config["heating_event_max_celsius"],
    precondition_hours=control_config["preconditioning_window_hours"],
)

heating_schedule_df, heating_diagnostics_df = optimize_event_control_schedules(
    df=df_h_optimizer,
    events_df=events_heating_validation,
    thermal_model_dict=model_h,
    device_power_model_dict=model_power_h,
    config=heating_event_config,
    candidate_offsets=control_config["candidate_offsets_minutes"],
    max_simultaneous_devices=control_config["max_simultaneous_devices"],
    electricity_price=energy_config["electricity_price_eur_per_kWh"],
    co2_factor=energy_config["co2_kg_per_kWh"],
    expected_devices=control_config["expected_devices"],
    merge_threshold_minutes=control_config["merge_threshold_minutes"],
    top_candidate_count=control_config["top_candidate_count"],
)

print("Heating selected event-control schedule:")
display(heating_schedule_df)
print("Heating compact top-candidate diagnostics:")
display(heating_diagnostics_df.head(15))
print("Heating feasibility summary:")
display(
    heating_schedule_df.groupby(["feasible", "selection_reason"])
    .size()
    .reset_index(name="event_count")
)

Heating selected event-control schedule:


,event_id,mode,start_time,device_1_on,device_2_on,device_3_on,device_4_on,target_temp,predicted_temp_at_start,expected_kwh,...,selection_reason,comfort_violation_c,candidate_start_offset_minutes,active_device_count,is_feasible,violation_celsius,window_id,window_starts_at,window_ends_at,window_rationale
0,E01,heating,2026-04-03 04:30:00,0,0,0,0,20.0,18.263571,0.0,...,infeasible fallback with smallest comfort viol...,1.736429,0,0,False,1.736429,W01,2026-04-03 04:30:00,2026-04-03 21:30:00,Merged 2 events due to overlap or close proxim...
1,E02,heating,2026-04-03 04:30:00,0,0,0,0,20.0,18.263571,0.0,...,infeasible fallback with smallest comfort viol...,1.736429,0,0,False,1.736429,W01,2026-04-03 04:30:00,2026-04-03 21:30:00,Merged 2 events due to overlap or close proxim...
2,E03,heating,2026-04-04 04:30:00,0,0,0,0,20.0,18.434000,0.0,...,infeasible fallback with smallest comfort viol...,1.566000,0,0,False,1.566000,W02,2026-04-04 04:30:00,2026-04-04 21:30:00,Merged 2 events due to overlap or close proxim...
3,E04,heating,2026-04-04 04:30:00,0,0,0,0,20.0,18.434000,0.0,...,infeasible fallback with smallest comfort viol...,1.566000,0,0,False,1.566000,W02,2026-04-04 04:30:00,2026-04-04 21:30:00,Merged 2 events due to overlap or close proxim...
4,E05,heating,2026-04-05 04:30:00,0,0,0,0,20.0,18.509000,0.0,...,infeasible fallback with smallest comfort viol...,1.491000,0,0,False,1.491000,W03,2026-04-05 04:30:00,2026-04-05 21:30:00,Merged 2 events due to overlap or close proxim...
5,E06,heating,2026-04-05 04:30:00,0,0,0,0,20.0,18.509000,0.0,...,infeasible fallback with smallest comfort viol...,1.491000,0,0,False,1.491000,W03,2026-04-05 04:30:00,2026-04-05 21:30:00,Merged 2 events due to overlap or close proxim...


Heating compact top-candidate diagnostics:


,window_id,candidate_rank,selected,feasible,selection_reason,comfort_violation_c,candidate_start_offset_minutes,active_device_count,active_devices,predicted_temp_at_start,expected_kwh,expected_cost,expected_co2,window_rationale,violation_celsius,is_feasible,offset_minutes,original_event_ids
0,W01,1,True,False,infeasible fallback with smallest comfort viol...,1.736429,0,0,,18.263571,0.0,0.0,0.0,Merged 2 events due to overlap or close proxim...,1.736429,False,0,"E01,E02"
1,W01,2,False,False,top candidate for comparison,1.736429,0,1,1,18.263571,0.0,0.0,0.0,Merged 2 events due to overlap or close proxim...,1.736429,False,0,"E01,E02"
2,W01,3,False,False,top candidate for comparison,1.736429,0,1,2,18.263571,0.0,0.0,0.0,Merged 2 events due to overlap or close proxim...,1.736429,False,0,"E01,E02"
3,W01,4,False,False,top candidate for comparison,1.736429,0,1,3,18.263571,0.0,0.0,0.0,Merged 2 events due to overlap or close proxim...,1.736429,False,0,"E01,E02"
4,W01,5,False,False,top candidate for comparison,1.736429,0,1,4,18.263571,0.0,0.0,0.0,Merged 2 events due to overlap or close proxim...,1.736429,False,0,"E01,E02"
5,W02,1,True,False,infeasible fallback with smallest comfort viol...,1.566000,0,0,,18.434000,0.0,0.0,0.0,Merged 2 events due to overlap or close proxim...,1.566000,False,0,"E03,E04"
6,W02,2,False,False,top candidate for comparison,1.566000,0,1,1,18.434000,0.0,0.0,0.0,Merged 2 events due to overlap or close proxim...,1.566000,False,0,"E03,E04"
7,W02,3,False,False,top candidate for comparison,1.566000,0,1,2,18.434000,0.0,0.0,0.0,Merged 2 events due to overlap or close proxim...,1.566000,False,0,"E03,E04"
8,W02,4,False,False,top candidate for comparison,1.566000,0,1,3,18.434000,0.0,0.0,0.0,Merged 2 events due to overlap or close proxim...,1.566000,False,0,"E03,E04"
9,W02,5,False,False,top candidate for comparison,1.566000,0,1,4,18.434000,0.0,0.0,0.0,Merged 2 events due to overlap or close proxim...,1.566000,False,0,"E03,E04"


Heating feasibility summary:


,feasible,selection_reason,event_count
0,False,infeasible fallback with smallest comfort viol...,6


### Optimize Cooling Validation Events


In [10]:
cooling_event_config = EventControlConfig(
    mode="cooling",
    min_temp=comfort_config["cooling_event_min_celsius"],
    max_temp=comfort_config["cooling_event_max_celsius"],
    precondition_hours=control_config["preconditioning_window_hours"],
)

cooling_schedule_df, cooling_diagnostics_df = optimize_event_control_schedules(
    df=df_c_optimizer,
    events_df=events_cooling_validation,
    thermal_model_dict=model_c,
    device_power_model_dict=model_power_c,
    config=cooling_event_config,
    candidate_offsets=control_config["candidate_offsets_minutes"],
    max_simultaneous_devices=control_config["max_simultaneous_devices"],
    electricity_price=energy_config["electricity_price_eur_per_kWh"],
    co2_factor=energy_config["co2_kg_per_kWh"],
    expected_devices=control_config["expected_devices"],
    merge_threshold_minutes=control_config["merge_threshold_minutes"],
    top_candidate_count=control_config["top_candidate_count"],
)

print("Cooling selected event-control schedule:")
display(cooling_schedule_df)
print("Cooling compact top-candidate diagnostics:")
display(cooling_diagnostics_df.head(15))
print("Cooling feasibility summary:")
display(
    cooling_schedule_df.groupby(["feasible", "selection_reason"])
    .size()
    .reset_index(name="event_count")
)

Cooling selected event-control schedule:


,event_id,mode,start_time,device_1_on,device_2_on,device_3_on,device_4_on,target_temp,predicted_temp_at_start,expected_kwh,...,selection_reason,comfort_violation_c,candidate_start_offset_minutes,active_device_count,is_feasible,violation_celsius,window_id,window_starts_at,window_ends_at,window_rationale
0,E01,cooling,2026-05-28 05:45:00,0,0,0,0,24.0,19.981858,1.466293,...,infeasible fallback with smallest comfort viol...,0.018142,15,0,False,0.018142,W01,2026-05-28 06:00:00,2026-05-28 18:00:00,Merged 7 events due to overlap or close proxim...
1,E02,cooling,2026-05-28 05:45:00,0,0,0,0,24.0,19.981858,1.466293,...,infeasible fallback with smallest comfort viol...,0.018142,15,0,False,0.018142,W01,2026-05-28 06:00:00,2026-05-28 18:00:00,Merged 7 events due to overlap or close proxim...
2,E03,cooling,2026-05-28 05:45:00,0,0,0,0,24.0,19.981858,1.466293,...,infeasible fallback with smallest comfort viol...,0.018142,15,0,False,0.018142,W01,2026-05-28 06:00:00,2026-05-28 18:00:00,Merged 7 events due to overlap or close proxim...
3,E04,cooling,2026-05-28 05:45:00,0,0,0,0,24.0,19.981858,1.466293,...,infeasible fallback with smallest comfort viol...,0.018142,15,0,False,0.018142,W01,2026-05-28 06:00:00,2026-05-28 18:00:00,Merged 7 events due to overlap or close proxim...
4,E05,cooling,2026-05-28 05:45:00,0,0,0,0,24.0,19.981858,1.466293,...,infeasible fallback with smallest comfort viol...,0.018142,15,0,False,0.018142,W01,2026-05-28 06:00:00,2026-05-28 18:00:00,Merged 7 events due to overlap or close proxim...
5,E06,cooling,2026-05-28 05:45:00,0,0,0,0,24.0,19.981858,1.466293,...,infeasible fallback with smallest comfort viol...,0.018142,15,0,False,0.018142,W01,2026-05-28 06:00:00,2026-05-28 18:00:00,Merged 7 events due to overlap or close proxim...
6,E07,cooling,2026-05-28 05:45:00,0,0,0,0,24.0,19.981858,1.466293,...,infeasible fallback with smallest comfort viol...,0.018142,15,0,False,0.018142,W01,2026-05-28 06:00:00,2026-05-28 18:00:00,Merged 7 events due to overlap or close proxim...
7,E08,cooling,2026-05-29 06:15:00,0,0,0,0,24.0,18.653493,7.331466,...,infeasible fallback with smallest comfort viol...,1.346507,75,0,False,1.346507,W02,2026-05-29 07:30:00,2026-05-29 10:00:00,Merged 3 events due to overlap or close proxim...
8,E09,cooling,2026-05-29 06:15:00,0,0,0,0,24.0,18.653493,7.331466,...,infeasible fallback with smallest comfort viol...,1.346507,75,0,False,1.346507,W02,2026-05-29 07:30:00,2026-05-29 10:00:00,Merged 3 events due to overlap or close proxim...
9,E10,cooling,2026-05-29 06:15:00,0,0,0,0,24.0,18.653493,7.331466,...,infeasible fallback with smallest comfort viol...,1.346507,75,0,False,1.346507,W02,2026-05-29 07:30:00,2026-05-29 10:00:00,Merged 3 events due to overlap or close proxim...


Cooling compact top-candidate diagnostics:


,window_id,candidate_rank,selected,feasible,selection_reason,comfort_violation_c,candidate_start_offset_minutes,active_device_count,active_devices,predicted_temp_at_start,expected_kwh,expected_cost,expected_co2,window_rationale,violation_celsius,is_feasible,offset_minutes,original_event_ids
0,W01,1,True,False,infeasible fallback with smallest comfort viol...,0.018142,15,0,,19.981858,1.466293,0.513203,0.586517,Merged 7 events due to overlap or close proxim...,0.018142,False,15,"E01,E02,E03,E04,E05,E06,E07"
1,W01,2,False,False,top candidate for comparison,0.024733,15,1,1,19.975267,1.624773,0.568671,0.649909,Merged 7 events due to overlap or close proxim...,0.024733,False,15,"E01,E02,E03,E04,E05,E06,E07"
2,W01,3,False,False,top candidate for comparison,0.041000,0,0,,19.959000,0.000000,0.000000,0.000000,Merged 7 events due to overlap or close proxim...,0.041000,False,0,"E01,E02,E03,E04,E05,E06,E07"
3,W01,4,False,False,top candidate for comparison,0.041000,0,1,1,19.959000,0.000000,0.000000,0.000000,Merged 7 events due to overlap or close proxim...,0.041000,False,0,"E01,E02,E03,E04,E05,E06,E07"
4,W01,5,False,False,top candidate for comparison,0.041000,0,1,2,19.959000,0.000000,0.000000,0.000000,Merged 7 events due to overlap or close proxim...,0.041000,False,0,"E01,E02,E03,E04,E05,E06,E07"
5,W02,1,True,False,infeasible fallback with smallest comfort viol...,1.346507,75,0,,18.653493,7.331466,2.566013,2.932586,Merged 3 events due to overlap or close proxim...,1.346507,False,75,"E08,E09,E10"
6,W02,2,False,False,top candidate for comparison,1.375984,75,1,1,18.624016,8.123865,2.843353,3.249546,Merged 3 events due to overlap or close proxim...,1.375984,False,75,"E08,E09,E10"
7,W02,3,False,False,top candidate for comparison,1.388111,60,0,,18.611889,5.865173,2.052810,2.346069,Merged 3 events due to overlap or close proxim...,1.388111,False,60,"E08,E09,E10"
8,W02,4,False,False,top candidate for comparison,1.390078,90,0,,18.609922,8.797759,3.079216,3.519104,Merged 3 events due to overlap or close proxim...,1.390078,False,90,"E08,E09,E10"
9,W02,5,False,False,top candidate for comparison,1.398310,105,0,,18.601690,10.264052,3.592418,4.105621,Merged 3 events due to overlap or close proxim...,1.398310,False,105,"E08,E09,E10"


Cooling feasibility summary:


,feasible,selection_reason,event_count
0,False,infeasible fallback with smallest comfort viol...,10


### Persist Selected Schedules


In [11]:
heating_report_path = save_event_control_schedule_report(
    heating_schedule_df,
    mode="heating",
    report_name="heating_event_control_results.csv",
)
cooling_report_path = save_event_control_schedule_report(
    cooling_schedule_df,
    mode="cooling",
    report_name="cooling_event_control_results.csv",
)

print(f"Heating event-control schedule report: {heating_report_path}")
print(f"Cooling event-control schedule report: {cooling_report_path}")

required_schedule_columns = [
    "event_id",
    "mode",
    "start_time",
    "device_1_on",
    "device_2_on",
    "device_3_on",
    "device_4_on",
    "target_temp",
    "predicted_temp_at_start",
    "expected_kwh",
    "expected_cost",
    "expected_co2",
    "feasible",
    "selection_reason",
    "comfort_violation_c",
    "candidate_start_offset_minutes",
    "active_device_count",
]

for name, schedule_df in [("heating", heating_schedule_df), ("cooling", cooling_schedule_df)]:
    missing = [col for col in required_schedule_columns if col not in schedule_df.columns]
    assert not missing, f"{name} schedule missing required columns: {missing}"
    assert not schedule_df.empty, f"{name} schedule unexpectedly empty"

print("Persisted selected schedule CSVs with required S02 output columns.")

Heating event-control schedule report: /home/jovyan/heatsync/backend/artifacts/event_device_control/reports/heating_event_control_results.csv
Cooling event-control schedule report: /home/jovyan/heatsync/backend/artifacts/event_device_control/reports/cooling_event_control_results.csv
Persisted selected schedule CSVs with required S02 output columns.


## 6. Baseline-vs-Smart-Control Validation Reports, Figures, and Executive Recap

Use the selected S02 schedules to build leakage-safe baseline-vs-smart-control time-series, overwrite the per-event reports with comparison metrics while preserving schedule columns, save judge-facing validation figures, and summarize savings, comfort, artifact paths, and edge cases.


In [12]:
from pathlib import Path

import matplotlib.pyplot as plt

reports_dir = Path("../artifacts/event_device_control/reports")
figures_dir = Path("../artifacts/event_device_control/figures")
reports_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

season_outputs = {}


def build_validation_outputs(
    *,
    mode,
    validation_df,
    schedule_df,
    device_power_model,
    comfort_min,
    comfort_max,
):
    comparison = build_event_control_comparison_timeseries(
        validation_df=validation_df,
        schedule_df=schedule_df,
        device_power_model_dict=device_power_model,
        electricity_price=energy_config["electricity_price_eur_per_kWh"],
        co2_factor=energy_config["co2_kg_per_kWh"],
        comfort_min=comfort_min,
        comfort_max=comfort_max,
        expected_devices=control_config["expected_devices"],
    )
    augmented_report = augment_event_control_report(
        schedule_df=schedule_df,
        comparison_df=comparison,
        comfort_min=comfort_min,
        comfort_max=comfort_max,
    )

    report_path = reports_dir / f"{mode}_event_control_results.csv"
    timeseries_path = reports_dir / f"{mode}_validation_timeseries.csv"
    augmented_report.to_csv(report_path, index=False)
    comparison.to_csv(timeseries_path, index=False)
    return comparison, augmented_report, report_path, timeseries_path


heating_timeseries_df, heating_event_report_df, heating_report_path, heating_timeseries_path = (
    build_validation_outputs(
        mode="heating",
        validation_df=df_h_optimizer,
        schedule_df=heating_schedule_df,
        device_power_model=model_power_h,
        comfort_min=comfort_config["heating_event_min_celsius"],
        comfort_max=comfort_config["heating_event_max_celsius"],
    )
)

cooling_timeseries_df, cooling_event_report_df, cooling_report_path, cooling_timeseries_path = (
    build_validation_outputs(
        mode="cooling",
        validation_df=df_c_optimizer,
        schedule_df=cooling_schedule_df,
        device_power_model=model_power_c,
        comfort_min=comfort_config["cooling_event_min_celsius"],
        comfort_max=comfort_config["cooling_event_max_celsius"],
    )
)

print("Saved augmented per-event reports and validation time-series:")
for label, path in [
    ("heating report", heating_report_path),
    ("heating time-series", heating_timeseries_path),
    ("cooling report", cooling_report_path),
    ("cooling time-series", cooling_timeseries_path),
]:
    print(f"  {label}: {path}")

required_metric_columns = EVENT_CONTROL_REPORT_METRIC_COLUMNS

for name, report_df, timeseries_df in [
    ("heating", heating_event_report_df, heating_timeseries_df),
    ("cooling", cooling_event_report_df, cooling_timeseries_df),
]:
    missing_report_columns = [
        col
        for col in required_schedule_columns + required_metric_columns
        if col not in report_df.columns
    ]
    assert not missing_report_columns, (
        f"{name} augmented report missing columns: {missing_report_columns}"
    )
    assert not report_df.empty, f"{name} augmented report unexpectedly empty"
    assert not timeseries_df.empty, f"{name} validation time-series unexpectedly empty"
    assert {
        "timestamp",
        "baseline_power_kw",
        "smart_power_kw",
        "cumulative_baseline_kwh",
        "cumulative_smart_kwh",
    }.issubset(timeseries_df.columns), f"{name} time-series missing comparison columns"

Saved augmented per-event reports and validation time-series:
  heating report: ../artifacts/event_device_control/reports/heating_event_control_results.csv
  heating time-series: ../artifacts/event_device_control/reports/heating_validation_timeseries.csv
  cooling report: ../artifacts/event_device_control/reports/cooling_event_control_results.csv
  cooling time-series: ../artifacts/event_device_control/reports/cooling_validation_timeseries.csv


In [13]:
def plot_validation_story(  # noqa: PLR0915
    mode, comparison_df, schedule_df, figure_path, *, comfort_min, comfort_max
):
    # Drop the DatetimeIndex inherited from the helper so timestamp is only a column label.
    plot_df = comparison_df.copy().reset_index(drop=True)
    plot_df["timestamp"] = pd.to_datetime(plot_df["timestamp"])
    plot_df = plot_df.sort_values("timestamp")

    device_cols = [
        col for col in schedule_df.columns if col.startswith("device_") and col.endswith("_on")
    ]
    palette = plt.get_cmap("tab10")

    fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True, constrained_layout=True)
    fig.suptitle(
        f"{mode.title()} validation: measured baseline vs smart control",
        fontsize=15,
        fontweight="bold",
    )

    axes[0].plot(
        plot_df["timestamp"],
        plot_df["room_temp"],
        color="tab:blue",
        linewidth=1.8,
        label="Measured room temp",
    )
    axes[0].fill_between(
        plot_df["timestamp"],
        comfort_min,
        comfort_max,
        color="tab:green",
        alpha=0.15,
        label="Comfort band",
    )
    axes[0].set_ylabel("Room temp (°C)")
    axes[0].legend(loc="upper right")
    axes[0].grid(True, alpha=0.25)

    axes[1].plot(
        plot_df["timestamp"],
        plot_df["baseline_power_kw"],
        color="tab:gray",
        linewidth=1.5,
        label="Measured baseline power",
    )
    axes[1].plot(
        plot_df["timestamp"],
        plot_df["smart_power_kw"],
        color="tab:orange",
        linewidth=1.5,
        label="Simulated smart-control power",
    )
    axes[1].set_ylabel("Power (kW)")
    axes[1].legend(loc="upper right")
    axes[1].grid(True, alpha=0.25)

    active_by_time = plot_df.set_index("timestamp")
    for idx, device_col in enumerate(device_cols):
        active = pd.Series(0.0, index=active_by_time.index)
        for _, schedule_row in schedule_df.iterrows():
            event_mask = (
                active_by_time["event_id"].astype(str).eq(str(schedule_row["event_id"]))
                if "event_id" in active_by_time.columns
                else pd.Series(False, index=active_by_time.index)
            )
            control_mask = (
                active_by_time["smart_control_active"].astype(bool)
                if "smart_control_active" in active_by_time.columns
                else pd.Series(False, index=active_by_time.index)
            )
            active.loc[event_mask & control_mask] = float(schedule_row.get(device_col, 0))
        axes[2].step(
            active.index,
            active + idx * 1.2,
            where="post",
            label=device_col,
            color=palette(idx),
            linewidth=1.4,
        )
    axes[2].set_ylabel("Device on/off")
    axes[2].set_yticks([idx * 1.2 for idx in range(len(device_cols))])
    axes[2].set_yticklabels(device_cols)
    axes[2].legend(loc="upper right", ncol=min(4, max(1, len(device_cols))))
    axes[2].grid(True, alpha=0.25)

    axes[3].plot(
        plot_df["timestamp"],
        plot_df["cumulative_baseline_kwh"],
        color="tab:gray",
        linewidth=1.8,
        label="Cumulative baseline kWh",
    )
    axes[3].plot(
        plot_df["timestamp"],
        plot_df["cumulative_smart_kwh"],
        color="tab:orange",
        linewidth=1.8,
        label="Cumulative smart kWh",
    )
    axes[3].set_ylabel("Energy (kWh)")
    axes[3].set_xlabel("Validation timestamp")
    axes[3].legend(loc="upper left")
    axes[3].grid(True, alpha=0.25)

    shaded = set()
    for _, schedule_row in schedule_df.iterrows():
        start = pd.Timestamp(schedule_row["start_time"])
        end = pd.Timestamp(schedule_row.get("window_ends_at", schedule_row["start_time"]))
        event_id = str(schedule_row["event_id"])
        if end <= start:
            matching = (
                plot_df[plot_df["event_id"].astype(str).eq(event_id)]
                if "event_id" in plot_df.columns
                else pd.DataFrame()
            )
            if not matching.empty:
                end = matching["timestamp"].max()
        label = "Event window / smart start" if not shaded else None
        for ax in axes:
            ax.axvspan(
                start, end, color="tab:red", alpha=0.06, label=label if ax is axes[0] else None
            )
            ax.axvline(start, color="tab:red", alpha=0.35, linestyle="--", linewidth=0.9)
        shaded.add(event_id)
    if shaded:
        axes[0].legend(loc="upper right")

    figure_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return figure_path


heating_figure_path = plot_validation_story(
    "heating",
    heating_timeseries_df,
    heating_event_report_df,
    figures_dir / "heating_validation_plot.png",
    comfort_min=comfort_config["heating_event_min_celsius"],
    comfort_max=comfort_config["heating_event_max_celsius"],
)

cooling_figure_path = plot_validation_story(
    "cooling",
    cooling_timeseries_df,
    cooling_event_report_df,
    figures_dir / "cooling_validation_plot.png",
    comfort_min=comfort_config["cooling_event_min_celsius"],
    comfort_max=comfort_config["cooling_event_max_celsius"],
)

for label, path in [
    ("heating figure", heating_figure_path),
    ("cooling figure", cooling_figure_path),
]:
    assert path.exists() and path.stat().st_size > 0, f"{label} was not written: {path}"
    print(f"{label}: {path} ({path.stat().st_size:,} bytes)")

heating figure: ../artifacts/event_device_control/figures/heating_validation_plot.png (325,996 bytes)
cooling figure: ../artifacts/event_device_control/figures/cooling_validation_plot.png (277,616 bytes)


In [14]:
def summarize_validation_report(
    mode, report_df, timeseries_df, report_path, timeseries_path, figure_path
):
    status_counts = report_df["report_status"].value_counts().to_dict()
    baseline_kwh = float(report_df["baseline_kwh"].sum())
    smart_kwh = float(report_df["smart_kwh"].sum())
    baseline_cost = float(report_df["baseline_cost_eur"].sum())
    smart_cost = float(report_df["smart_cost_eur"].sum())
    baseline_co2 = float(report_df["baseline_co2_kg"].sum())
    smart_co2 = float(report_df["smart_co2_kg"].sum())
    return {
        "mode": mode,
        "events": len(report_df),
        "baseline_kwh": baseline_kwh,
        "smart_kwh": smart_kwh,
        "energy_savings_kwh": baseline_kwh - smart_kwh,
        "energy_savings_pct": 0.0
        if baseline_kwh == 0
        else (baseline_kwh - smart_kwh) / baseline_kwh * 100.0,
        "baseline_cost_eur": baseline_cost,
        "smart_cost_eur": smart_cost,
        "cost_savings_eur": baseline_cost - smart_cost,
        "cost_savings_pct": 0.0
        if baseline_cost == 0
        else (baseline_cost - smart_cost) / baseline_cost * 100.0,
        "co2_savings_kg": baseline_co2 - smart_co2,
        "comfort_hit_rate_pct": float(timeseries_df["inside_comfort_band"].mean() * 100.0),
        "degree_minutes_violation": float(timeseries_df["degree_minutes_violation"].sum()),
        "infeasible_events": int(status_counts.get("infeasible", 0)),
        "comfort_missed_events": int(status_counts.get("comfort_missed", 0)),
        "neutral_savings_events": int(status_counts.get("neutral_savings", 0)),
        "negative_savings_events": int(status_counts.get("negative_savings", 0)),
        "per_event_csv": str(report_path),
        "timeseries_csv": str(timeseries_path),
        "figure_png": str(figure_path),
    }


executive_recap_df = pd.DataFrame(
    [
        summarize_validation_report(
            "heating",
            heating_event_report_df,
            heating_timeseries_df,
            heating_report_path,
            heating_timeseries_path,
            heating_figure_path,
        ),
        summarize_validation_report(
            "cooling",
            cooling_event_report_df,
            cooling_timeseries_df,
            cooling_report_path,
            cooling_timeseries_path,
            cooling_figure_path,
        ),
    ]
)

savings_columns = [
    "mode",
    "events",
    "baseline_kwh",
    "smart_kwh",
    "energy_savings_kwh",
    "energy_savings_pct",
    "baseline_cost_eur",
    "smart_cost_eur",
    "cost_savings_eur",
    "cost_savings_pct",
    "co2_savings_kg",
]
comfort_columns = [
    "mode",
    "comfort_hit_rate_pct",
    "degree_minutes_violation",
    "infeasible_events",
    "comfort_missed_events",
    "neutral_savings_events",
    "negative_savings_events",
]
path_columns = ["mode", "per_event_csv", "timeseries_csv", "figure_png"]

print("Executive recap — savings and impact:")
display(executive_recap_df[savings_columns].round(3))
print("Executive recap — comfort and edge cases:")
display(executive_recap_df[comfort_columns].round(3))
print("Executive recap — saved artifact paths:")
display(executive_recap_df[path_columns])

expected_outputs = [
    heating_report_path,
    cooling_report_path,
    heating_timeseries_path,
    cooling_timeseries_path,
    heating_figure_path,
    cooling_figure_path,
]
missing_outputs = [str(path) for path in expected_outputs if not Path(path).exists()]
assert not missing_outputs, f"Missing validation artifacts: {missing_outputs}"
print("All S03 report and figure artifacts are present.")

Executive recap — savings and impact:


,mode,events,baseline_kwh,smart_kwh,energy_savings_kwh,energy_savings_pct,baseline_cost_eur,smart_cost_eur,cost_savings_eur,cost_savings_pct,co2_savings_kg
0,heating,6,2610.382,479.104,2131.278,81.646,913.634,167.686,745.947,81.646,852.511
1,cooling,10,1515.817,568.922,946.895,62.468,530.536,199.123,331.413,62.468,378.758


Executive recap — comfort and edge cases:


,mode,comfort_hit_rate_pct,degree_minutes_violation,infeasible_events,comfort_missed_events,neutral_savings_events,negative_savings_events
0,heating,15.571,4504.920,6,0,0,0
1,cooling,72.468,2162.184,10,0,0,0


Executive recap — saved artifact paths:


,mode,per_event_csv,timeseries_csv,figure_png
0,heating,../artifacts/event_device_control/reports/heat...,../artifacts/event_device_control/reports/heat...,../artifacts/event_device_control/figures/heat...
1,cooling,../artifacts/event_device_control/reports/cool...,../artifacts/event_device_control/reports/cool...,../artifacts/event_device_control/figures/cool...


All S03 report and figure artifacts are present.
